# UQ Verification: MC Dropout vs Deep Ensemble

**Goal:** Re-run uncertainty quantification on existing trained models to verify all results.  
**Models:** T8 PointNetTransfGAT (MC Dropout) + 5-seed Deep Ensemble  
**Test set:** 100 graphs from `test_dl.pt` (same deterministic split, seed=42)  

**Output saved to Drive:**
```
TR-C_Benchmarks/uq_verification_run/
  mc_dropout_verified.npz
  mc_dropout_verified_metrics.json
  ensemble_verified.npz
  ensemble_verified_metrics.json
  comparison_verified.json
```

## Cell 1 — Install PyTorch Geometric

In [ ]:
import subprocess, sys, torch

TORCH = torch.__version__.split('+')[0]
CUDA  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f"torch={TORCH}  cuda_tag={CUDA}")

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    f'torch-scatter', '-f', f'https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    f'torch-sparse',  '-f', f'https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html'])
print('Done.')

## Cell 2 — Mount Drive + Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, math
import numpy as np
import torch
import torch.nn as nn
from torch_geometric.nn import (
    Sequential as GeoSequential,
    TransformerConv, GATConv, PointNetConv
)
from torch_geometric.data import Batch
from scipy.stats import spearmanr

print('All imports OK')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

## Cell 3 — Config
> **Only edit this cell if your Drive paths differ.**

In [ ]:
# ── PATHS ──────────────────────────────────────────────────────────────
DRIVE_BASE  = '/content/drive/MyDrive/data/TR-C_Benchmarks'
T8_DIR      = f'{DRIVE_BASE}/point_net_transf_gat_8th_trial_lower_dropout'
T8_MODEL    = f'{T8_DIR}/trained_model/model.pth'
TEST_DL     = f'{T8_DIR}/data_created_during_training/test_dl.pt'

ENS_SEEDS   = [42, 137, 256, 389, 512]
ENS_MODELS  = {
    seed: f'{DRIVE_BASE}/deep_ensemble_seed{seed}/trained_model/model.pth'
    for seed in ENS_SEEDS
}

MC_SAMPLES  = 30
OUT_DIR     = f'{DRIVE_BASE}/uq_verification_run'
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── VERIFY PATHS ───────────────────────────────────────────────────────
print(f'Device         : {DEVICE}')
print(f'Output dir     : {OUT_DIR}')
print()
ok = True
for label, path in [('T8 model.pth', T8_MODEL), ('test_dl.pt', TEST_DL)]:
    exists = os.path.exists(path)
    print(f'  {"OK" if exists else "MISSING"} | {label}: {path}')
    if not exists: ok = False
for seed, path in ENS_MODELS.items():
    exists = os.path.exists(path)
    print(f'  {"OK" if exists else "MISSING"} | seed {seed}: {path}')
    if not exists: ok = False
print()
print('All paths verified.' if ok else 'FIX MISSING PATHS ABOVE before continuing.')

## Cell 3b — Upload Missing Ensemble Models to Drive

> **Run this cell only if ensemble seeds showed MISSING above.**  
> Upload the 5 `model.pth` files from your local PC.  
> Files needed (from your local project):  
> - `code/data/TR-C_Benchmarks/deep_ensemble_seed42/trained_model/model.pth`  
> - `code/data/TR-C_Benchmarks/deep_ensemble_seed137/trained_model/model.pth`  
> - `code/data/TR-C_Benchmarks/deep_ensemble_seed256/trained_model/model.pth`  
> - `code/data/TR-C_Benchmarks/deep_ensemble_seed389/trained_model/model.pth`  
> - `code/data/TR-C_Benchmarks/deep_ensemble_seed512/trained_model/model.pth`

In [ ]:
# ── Step 1: Show what's on Drive (to debug / find actual paths) ─────────
import os
print('Contents of TR-C_Benchmarks on Drive:')
for entry in sorted(os.listdir(DRIVE_BASE)):
    full = os.path.join(DRIVE_BASE, entry)
    tag  = '[DIR]' if os.path.isdir(full) else '[FILE]'
    print(f'  {tag} {entry}')

print()
print('If deep_ensemble_seed* folders are missing, run Step 2 below.')

In [ ]:
# ── Step 2: Upload 5 model.pth files from local PC ──────────────────────
# A file picker will open — select ALL 5 model.pth files at once.
# Then this code saves them to the correct Drive paths automatically.
#
# Local paths on your PC:
#   ml_surrogates_thesis_final/code/data/TR-C_Benchmarks/
#     deep_ensemble_seed42/trained_model/model.pth
#     deep_ensemble_seed137/trained_model/model.pth
#     deep_ensemble_seed256/trained_model/model.pth
#     deep_ensemble_seed389/trained_model/model.pth
#     deep_ensemble_seed512/trained_model/model.pth
#
# NOTE: All 5 files are named model.pth — rename them before uploading:
#   model_seed42.pth, model_seed137.pth, model_seed256.pth,
#   model_seed389.pth, model_seed512.pth

import shutil
from google.colab import files

SEED_MAP = {
    'model_seed42.pth' : 42,
    'model_seed137.pth': 137,
    'model_seed256.pth': 256,
    'model_seed389.pth': 389,
    'model_seed512.pth': 512,
}

print('Select your renamed model files (model_seed42.pth, etc.)...')
uploaded = files.upload()   # opens file picker

for filename, content in uploaded.items():
    seed = SEED_MAP.get(filename)
    if seed is None:
        print(f'  SKIP (unrecognised filename): {filename}')
        continue
    dest_dir  = f'{DRIVE_BASE}/deep_ensemble_seed{seed}/trained_model'
    dest_path = f'{dest_dir}/model.pth'
    os.makedirs(dest_dir, exist_ok=True)
    with open(dest_path, 'wb') as f:
        f.write(content)
    print(f'  Saved seed {seed} -> {dest_path}  ({len(content)/1e6:.1f} MB)')

# Re-verify paths
print()
print('Re-checking ensemble paths...')
all_ok = True
for seed, path in ENS_MODELS.items():
    exists = os.path.exists(path)
    print(f'  {"OK" if exists else "MISSING"} | seed {seed}')
    if not exists: all_ok = False
print('All ensemble models ready!' if all_ok else 'Still missing — check filenames.')

## Cell 4 — PointNetTransfGAT Model Definition

In [ ]:
import torch.nn.init as init
from torch_geometric.nn import GATConv as _GAT, PointNetConv as _PNC

class BaseGNN(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.3,
                 use_dropout=False, predict_mode_stats=False,
                 dtype=torch.float32, log_to_wandb=False):
        super().__init__()
        self.in_channels        = in_channels
        self.out_channels       = out_channels
        self.dropout            = dropout
        self.use_dropout        = use_dropout
        self.predict_mode_stats = predict_mode_stats
        self.dtype              = dtype
        self.log_to_wandb       = log_to_wandb

    def initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.zeros_(m.bias)


class PointNetTransfGAT(BaseGNN):
    def __init__(self,
                 in_channels=5, out_channels=1,
                 point_net_conv_layer_structure_local_mlp=None,
                 point_net_conv_layer_structure_global_mlp=None,
                 gat_conv_layer_structure=None,
                 dropout=0.3, use_dropout=False,
                 predict_mode_stats=False,
                 dtype=torch.float32, log_to_wandb=False):
        super().__init__(in_channels, out_channels, dropout, use_dropout,
                         predict_mode_stats, dtype, log_to_wandb)
        self.pnc_local  = point_net_conv_layer_structure_local_mlp  or [256]
        self.pnc_global = point_net_conv_layer_structure_global_mlp or [512]
        self.gat_conv   = gat_conv_layer_structure                   or [128, 256, 512]
        self.define_layers()
        self.initialize_weights()

    # ── layers ──────────────────────────────────────────────────────────
    def define_layers(self):
        if self.use_dropout:
            self.dropout_layer = nn.Dropout(self.dropout)
        self.point_net_conv_1 = self._make_pnc(is_first=True,  is_last=False)
        self.point_net_conv_2 = self._make_pnc(is_first=False, is_last=True)
        self.gat_graph_layers = GeoSequential('x, edge_index', self._gat_layers())
        self.gat_final        = GATConv(64, 1)

    def _gat_layers(self):
        layers = []
        for i in range(len(self.gat_conv) - 1):
            layers.append((
                TransformerConv(self.gat_conv[i], self.gat_conv[i+1] // 4, heads=4),
                'x, edge_index -> x'))
            layers.append(nn.ReLU(inplace=True))
            if self.use_dropout:
                layers.append(self.dropout_layer)
        layers.append((GATConv(self.gat_conv[-1], 64), 'x, edge_index -> x'))
        return layers

    def _make_pnc(self, is_first, is_last):
        offset   = 2
        local_in = (self.in_channels if is_first else self.pnc_global[-1]) + offset

        local_layers = [nn.Linear(local_in, self.pnc_local[0]), nn.ReLU()]
        if self.use_dropout: local_layers.append(self.dropout_layer)
        for i in range(len(self.pnc_local) - 1):
            local_layers += [nn.Linear(self.pnc_local[i], self.pnc_local[i+1]), nn.ReLU()]
            if self.use_dropout: local_layers.append(self.dropout_layer)
        local_mlp = nn.Sequential(*local_layers)

        global_layers = [nn.Linear(self.pnc_local[-1], self.pnc_global[0]), nn.ReLU()]
        if self.use_dropout: global_layers.append(self.dropout_layer)
        for i in range(len(self.pnc_global) - 1):
            global_layers += [nn.Linear(self.pnc_global[i], self.pnc_global[i+1]), nn.ReLU()]
            if self.use_dropout: global_layers.append(self.dropout_layer)
        out_size = self.gat_conv[0] if is_last else self.pnc_global[-1]
        global_layers += [nn.Linear(self.pnc_global[-1], out_size), nn.ReLU()]
        if self.use_dropout: global_layers.append(self.dropout_layer)
        global_mlp = nn.Sequential(*global_layers)

        return _PNC(local_nn=local_mlp, global_nn=global_mlp)

    # ── forward ─────────────────────────────────────────────────────────
    def forward(self, data):
        x          = data.x.to(self.dtype)
        edge_index = data.edge_index
        pos1 = data.pos[:, 0, :]    # start position
        pos2 = data.pos[:, 1, :]    # end position
        x = self.point_net_conv_1(x, pos1, edge_index)
        x = self.point_net_conv_2(x, pos2, edge_index)
        x = self.gat_graph_layers(x, edge_index)
        return self.gat_final(x, edge_index)

    # ── weight init ─────────────────────────────────────────────────────
    def initialize_weights(self):
        super().initialize_weights()
        for m in self.modules():
            if isinstance(m, _PNC):
                for _, p in list(m.local_nn.named_parameters()) + \
                            list(m.global_nn.named_parameters()):
                    if p.dim() > 1: init.kaiming_normal_(p, mode='fan_out', nonlinearity='relu')
                    else:           init.zeros_(p)
            elif isinstance(m, _GAT):
                if hasattr(m, 'lin') and m.lin is not None:
                    init.xavier_normal_(m.lin.weight)
                    if m.lin.bias is not None: init.zeros_(m.lin.bias)
                for attr in ('att_src', 'att_dst'):
                    t = getattr(m, attr, None)
                    if t is not None: init.xavier_normal_(t)


# Quick sanity check
total_params = sum(p.numel() for p in
    PointNetTransfGAT(in_channels=5, out_channels=1,
                      use_dropout=True, dropout=0.2).parameters())
print(f'PointNetTransfGAT defined OK  |  params: {total_params:,}')

## Cell 5 — Helper Functions (mc_dropout_predict + metrics)

In [ ]:
def mc_dropout_predict(model, data, num_samples=30, device=None):
    """30 stochastic forward passes. Dropout ON, BatchNorm frozen."""
    was_training = model.training
    model = model.to(device)
    data  = data.to(device)

    model.train()   # dropout ON

    # freeze any BatchNorm running stats
    bn_states = []
    for m in model.modules():
        if isinstance(m, nn.modules.batchnorm._BatchNorm):
            bn_states.append((m, m.training))
            m.eval()

    preds = []
    with torch.inference_mode():
        for _ in range(num_samples):
            out = model(data)
            if isinstance(out, tuple): out = out[0]
            preds.append(out.detach())

    preds      = torch.stack(preds, dim=0)          # [S, N, 1]
    mean_pred  = preds.mean(dim=0)                  # [N, 1]
    uncertainty = preds.std(dim=0, unbiased=False)  # population std

    # restore model state
    model.train(was_training)
    for m, was_train in bn_states:
        m.train(was_train)

    return mean_pred.cpu().numpy(), uncertainty.cpu().numpy()


def compute_k_for_coverage(errors, uncertainties, target_coverage):
    """Find k s.t. |error| <= k*sigma achieves target_coverage fraction."""
    ratios = np.abs(errors) / np.clip(uncertainties, 1e-10, None)
    return float(np.percentile(ratios, target_coverage * 100))


def compute_coverage(errors, uncertainties, k):
    return float(np.mean(np.abs(errors) <= k * uncertainties))


def compute_all_metrics(preds, targets, uncertainties, graph_sizes, method_name):
    """Compute full UQ metric suite identical to evaluate_deep_ensemble.py."""
    preds         = preds.flatten().astype(np.float64)
    targets       = targets.flatten().astype(np.float64)
    uncertainties = uncertainties.flatten().astype(np.float64)

    errors     = targets - preds
    abs_errors = np.abs(errors)

    ss_res = np.sum(errors ** 2)
    ss_tot = np.sum((targets - np.mean(targets)) ** 2)
    r2   = float(1 - ss_res / ss_tot)
    mae  = float(np.mean(abs_errors))
    rmse = float(np.sqrt(np.mean(errors ** 2)))

    rho, rho_p = spearmanr(uncertainties, abs_errors)

    k95     = compute_k_for_coverage(errors, uncertainties, 0.95)
    k90     = compute_k_for_coverage(errors, uncertainties, 0.90)
    cov_196 = compute_coverage(errors, uncertainties, 1.96)

    coverages = {}
    for nom in [0.50, 0.70, 0.80, 0.90, 0.95]:
        k_nom = compute_k_for_coverage(errors, uncertainties, nom)
        coverages[f'k_{int(nom*100)}']            = k_nom
        coverages[f'cov_{int(nom*100)}_achieved'] = compute_coverage(errors, uncertainties, k_nom)

    # per-graph Spearman rho
    graph_starts = [sum(graph_sizes[:g]) for g in range(len(graph_sizes))]
    per_graph_rho = []
    for g in range(len(graph_sizes)):
        s, e  = graph_starts[g], graph_starts[g] + graph_sizes[g]
        rho_g, _ = spearmanr(uncertainties[s:e], abs_errors[s:e])
        if not np.isnan(rho_g):
            per_graph_rho.append(float(rho_g))
    per_graph_rho = np.array(per_graph_rho)

    return {
        'method'         : method_name,
        'n_test_graphs'  : len(graph_sizes),
        'n_total_nodes'  : int(len(preds)),
        'point_prediction': {'r2': r2, 'mae': mae, 'rmse': rmse},
        'uncertainty_quality': {
            'spearman_rho'    : float(rho),
            'spearman_p_value': float(rho_p),
            'mean_sigma'      : float(np.mean(uncertainties)),
            'std_sigma'       : float(np.std(uncertainties)),
            'min_sigma'       : float(np.min(uncertainties)),
            'max_sigma'       : float(np.max(uncertainties)),
        },
        'calibration': {
            'k95'                   : k95,
            'k90'                   : k90,
            'coverage_at_1.96sigma' : cov_196,
            **coverages,
        },
        'per_graph': {
            'mean_rho'      : float(np.mean(per_graph_rho)),
            'std_rho'       : float(np.std(per_graph_rho)),
            'min_rho'       : float(np.min(per_graph_rho)),
            'max_rho'       : float(np.max(per_graph_rho)),
            'n_valid_graphs': int(len(per_graph_rho)),
        },
    }


print('Helper functions defined OK')

## Cell 6 — Load Test Data

In [ ]:
print('Loading test_dl.pt ...')
t0 = time.time()
test_dataset = torch.load(TEST_DL, weights_only=False)
n_graphs     = len(test_dataset)
graph_sizes  = [test_dataset[g].x.shape[0] for g in range(n_graphs)]
total_nodes  = sum(graph_sizes)
print(f'  Loaded in {time.time()-t0:.1f}s')
print(f'  Graphs       : {n_graphs}')
print(f'  Nodes/graph  : {min(graph_sizes)} - {max(graph_sizes)}  (mean {total_nodes // n_graphs})')
print(f'  Total nodes  : {total_nodes:,}')
print(f'  x features   : {test_dataset[0].x.shape[1]}')
print(f'  y shape      : {test_dataset[0].y.shape}')
print(f'  pos shape    : {test_dataset[0].pos.shape}')

## Cell 7 — MC Dropout Evaluation
> T8 model (dropout=0.2), 30 stochastic samples per graph.  
> Expected time on A100: **~3-5 minutes**

In [ ]:
print('=' * 60)
print('  MC DROPOUT EVALUATION  (T8, dropout=0.2, 30 samples)')
print('=' * 60)

# ── Load model ──────────────────────────────────────────────────────────
print('\n[1/3] Loading T8 model...')
t8_model = PointNetTransfGAT(
    in_channels=5, out_channels=1,
    use_dropout=True, dropout=0.2
)
state = torch.load(T8_MODEL, map_location=DEVICE, weights_only=True)
t8_model.load_state_dict(state)
t8_model = t8_model.to(DEVICE)
n_params = sum(p.numel() for p in t8_model.parameters())
print(f'  Loaded on {DEVICE}  |  params: {n_params:,}')

# ── Run MC Dropout inference ─────────────────────────────────────────────
print(f'\n[2/3] Running MC Dropout ({MC_SAMPLES} samples x {n_graphs} graphs)...')
all_means, all_stds, all_targets = [], [], []
t0 = time.time()

for g_idx in range(n_graphs):
    graph = test_dataset[g_idx]
    mean, std = mc_dropout_predict(t8_model, graph, num_samples=MC_SAMPLES, device=DEVICE)
    all_means.append(mean.flatten())
    all_stds.append(std.flatten())
    all_targets.append(graph.y.cpu().numpy().flatten())

    if (g_idx + 1) % 10 == 0 or g_idx == 0:
        elapsed = time.time() - t0
        eta     = elapsed / (g_idx + 1) * (n_graphs - g_idx - 1)
        print(f'  Graph {g_idx+1:3d}/{n_graphs}  |  {elapsed:5.1f}s elapsed  |  ETA {eta:5.1f}s')

mc_preds   = np.concatenate(all_means)
mc_stds    = np.concatenate(all_stds)
mc_targets = np.concatenate(all_targets)
print(f'\n  Inference done in {time.time()-t0:.1f}s')
print(f'  sigma range: [{mc_stds.min():.4f}, {mc_stds.max():.4f}]')

# ── Compute metrics ──────────────────────────────────────────────────────
print('\n[3/3] Computing metrics...')
mc_results = compute_all_metrics(
    mc_preds, mc_targets, mc_stds, graph_sizes,
    f'MC Dropout (T8, dropout=0.2, {MC_SAMPLES} samples)'
)

# ── Save ─────────────────────────────────────────────────────────────────
np.savez(f'{OUT_DIR}/mc_dropout_verified.npz',
         predictions=mc_preds, uncertainties=mc_stds, targets=mc_targets)
with open(f'{OUT_DIR}/mc_dropout_verified_metrics.json', 'w') as f:
    json.dump(mc_results, f, indent=2)

# ── Print summary ────────────────────────────────────────────────────────
pp = mc_results['point_prediction']
uq = mc_results['uncertainty_quality']
ca = mc_results['calibration']
pg = mc_results['per_graph']
print(f'''
  POINT PREDICTION:
    R2   = {pp["r2"]:.4f}
    MAE  = {pp["mae"]:.3f} veh/h
    RMSE = {pp["rmse"]:.3f} veh/h

  UNCERTAINTY QUALITY:
    Spearman rho       = {uq["spearman_rho"]:.4f}  (p={uq["spearman_p_value"]:.2e})
    Mean sigma         = {uq["mean_sigma"]:.3f} veh/h

  CALIBRATION:
    k95                = {ca["k95"]:.2f}
    k90                = {ca["k90"]:.2f}
    Coverage @ 1.96sig = {ca["coverage_at_1.96sigma"]*100:.1f}%

  PER-GRAPH Spearman rho:
    Mean = {pg["mean_rho"]:.3f}  Std = {pg["std_rho"]:.3f}
    Range = [{pg["min_rho"]:.3f}, {pg["max_rho"]:.3f}]  ({pg["n_valid_graphs"]} graphs)

  Saved: {OUT_DIR}/mc_dropout_verified_metrics.json
''')

## Cell 8 — Deep Ensemble Evaluation
> 5 independently trained models (seeds 42/137/256/389/512), deterministic inference (dropout OFF).  
> Expected time on A100: **~1-2 minutes**

In [ ]:
print('=' * 60)
print('  DEEP ENSEMBLE EVALUATION  (5 seeds, deterministic)')
print('=' * 60)

member_preds_list = []
ens_targets       = None
ss_tot_ref        = None

for i, seed in enumerate(ENS_SEEDS):
    print(f'\n[{i+1}/{len(ENS_SEEDS)}] Loading seed {seed}...')
    t0 = time.time()

    model = PointNetTransfGAT(
        in_channels=5, out_channels=1,
        use_dropout=True, dropout=0.2
    )
    state = torch.load(ENS_MODELS[seed], map_location=DEVICE, weights_only=True)
    model.load_state_dict(state)
    model = model.to(DEVICE)
    model.eval()    # dropout OFF — fully deterministic

    preds_i, tgts_i = [], []
    with torch.no_grad():
        for g_idx in range(n_graphs):
            graph = test_dataset[g_idx].to(DEVICE)
            out   = model(graph).squeeze().cpu().numpy().flatten()
            preds_i.append(out)
            tgts_i.append(graph.y.cpu().numpy().flatten())

    preds_i = np.concatenate(preds_i)
    tgts_i  = np.concatenate(tgts_i)

    # verify all seeds see identical test targets
    if ens_targets is None:
        ens_targets = tgts_i
        ss_tot_ref  = np.sum((ens_targets - np.mean(ens_targets)) ** 2)
    else:
        assert np.allclose(ens_targets, tgts_i, atol=1e-4), \
            f'Target mismatch for seed {seed} — data split may differ!'

    r2_mem = float(1 - np.sum((tgts_i - preds_i) ** 2) / ss_tot_ref)
    print(f'  Seed {seed}  |  R2 = {r2_mem:.4f}  |  {time.time()-t0:.1f}s')

    member_preds_list.append(preds_i)
    del model
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

# ── Ensemble statistics ───────────────────────────────────────────────────
member_preds = np.stack(member_preds_list)              # [M, N]
ens_mean     = member_preds.mean(axis=0)
ens_std      = member_preds.std(axis=0, ddof=0)         # population std (ddof=0)

print(f'\nComputing ensemble metrics (M={len(ENS_SEEDS)}, N={len(ens_mean):,})...')
ens_results = compute_all_metrics(
    ens_mean, ens_targets, ens_std, graph_sizes,
    f'Deep Ensemble (M={len(ENS_SEEDS)}, seeds={ENS_SEEDS}, deterministic)'
)

# add per-member R2
member_r2 = {}
for seed, mp in zip(ENS_SEEDS, member_preds_list):
    member_r2[f'seed_{seed}'] = float(1 - np.sum((ens_targets - mp) ** 2) / ss_tot_ref)
ens_results['member_r2'] = member_r2

# ── Save ──────────────────────────────────────────────────────────────────
np.savez(f'{OUT_DIR}/ensemble_verified.npz',
         ensemble_mean=ens_mean, ensemble_std=ens_std,
         targets=ens_targets, member_predictions=member_preds)
with open(f'{OUT_DIR}/ensemble_verified_metrics.json', 'w') as f:
    json.dump(ens_results, f, indent=2)

# ── Print summary ─────────────────────────────────────────────────────────
pp = ens_results['point_prediction']
uq = ens_results['uncertainty_quality']
ca = ens_results['calibration']
pg = ens_results['per_graph']
print(f'''
  POINT PREDICTION:
    R2   = {pp["r2"]:.4f}
    MAE  = {pp["mae"]:.3f} veh/h
    RMSE = {pp["rmse"]:.3f} veh/h

  INDIVIDUAL MEMBER R2:''')
for s, r2v in member_r2.items():
    print(f'    {s}: {r2v:.4f}')
print(f'''
  UNCERTAINTY QUALITY:
    Spearman rho       = {uq["spearman_rho"]:.4f}  (p={uq["spearman_p_value"]:.2e})
    Mean sigma         = {uq["mean_sigma"]:.3f} veh/h

  CALIBRATION:
    k95                = {ca["k95"]:.2f}
    k90                = {ca["k90"]:.2f}
    Coverage @ 1.96sig = {ca["coverage_at_1.96sigma"]*100:.1f}%

  PER-GRAPH Spearman rho:
    Mean = {pg["mean_rho"]:.3f}  Std = {pg["std_rho"]:.3f}
    Range = [{pg["min_rho"]:.3f}, {pg["max_rho"]:.3f}]  ({pg["n_valid_graphs"]} graphs)

  Saved: {OUT_DIR}/ensemble_verified_metrics.json
''')

## Cell 9 — Final Comparison Table

In [ ]:
mc  = mc_results
ens = ens_results

print('\n' + '=' * 72)
print('  UQ VERIFICATION RESULTS — MC DROPOUT vs DEEP ENSEMBLE')
print('=' * 72)
print(f'  Test set: {n_graphs} graphs, {total_nodes:,} total nodes')
print(f'  Run at  : {time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime())}')
print()

rows = [
    ('R2',                'point_prediction',    'r2',                    '{:.4f}', 'higher = better'),
    ('MAE  (veh/h)',      'point_prediction',    'mae',                   '{:.3f}', 'lower = better'),
    ('RMSE (veh/h)',      'point_prediction',    'rmse',                  '{:.3f}', 'lower = better'),
    ('Spearman rho',      'uncertainty_quality', 'spearman_rho',          '{:.4f}', 'higher = sigma tracks error better'),
    ('Mean sigma (veh/h)','uncertainty_quality', 'mean_sigma',            '{:.3f}', 'informational'),
    ('k95',               'calibration',         'k95',                   '{:.2f}', 'lower = sharper intervals'),
    ('k90',               'calibration',         'k90',                   '{:.2f}', 'lower = sharper intervals'),
    ('Coverage@1.96sig',  'calibration',         'coverage_at_1.96sigma', '{:.3f}', 'ideal = 0.950'),
    ('Per-graph rho',     'per_graph',            'mean_rho',              '{:.3f}', 'higher = better local ranking'),
]

print(f'  {"Metric":<24} {"MC Dropout":>13} {"Ensemble":>13}  Note')
print('  ' + '-' * 68)
for name, section, key, fmt, note in rows:
    mc_val  = fmt.format(mc[section][key])
    ens_val = fmt.format(ens[section][key])
    print(f'  {name:<24} {mc_val:>13} {ens_val:>13}  {note}')

print('\n  INDIVIDUAL MEMBER R2:')
for s, r2v in ens.get('member_r2', {}).items():
    print(f'    {s}: {r2v:.4f}')

# ── Save comparison ───────────────────────────────────────────────────────
comparison = {
    'verified_at'  : time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime()),
    'test_set'     : {'n_graphs': n_graphs, 'total_nodes': total_nodes},
    'mc_dropout'   : {k: mc[k]  for k in ['point_prediction','uncertainty_quality','calibration','per_graph']},
    'deep_ensemble': {k: ens[k] for k in ['point_prediction','uncertainty_quality','calibration','per_graph']},
    'member_r2'    : ens.get('member_r2', {}),
}
with open(f'{OUT_DIR}/comparison_verified.json', 'w') as f:
    json.dump(comparison, f, indent=2)

print(f'\n  All results saved to: {OUT_DIR}/')
print(f'    mc_dropout_verified.npz')
print(f'    mc_dropout_verified_metrics.json')
print(f'    ensemble_verified.npz')
print(f'    ensemble_verified_metrics.json')
print(f'    comparison_verified.json')

## Cell 10 — Thesis Plots
> Generates 4 publication-quality figures and saves them to Drive.  
> **Requires Cells 7 & 8 to have run first** (uses in-memory arrays).
>
> Saved to `uq_verification_run/plots/`:
> - `fig1_scatter_predicted_vs_actual.png`
> - `fig2_calibration_curve.png`
> - `fig3_sigma_vs_error.png`
> - `fig4_uncertainty_distribution.png`

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

PLOT_DIR = f'{OUT_DIR}/plots'
os.makedirs(PLOT_DIR, exist_ok=True)

# ── Style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 10,
    'axes.titlesize'   : 11,
    'axes.labelsize'   : 10,
    'xtick.labelsize'  : 9,
    'ytick.labelsize'  : 9,
    'legend.fontsize'  : 9,
    'figure.dpi'       : 150,
    'savefig.dpi'      : 300,
    'savefig.bbox'     : 'tight',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})

MC_COLOR  = '#2c7bb6'
ENS_COLOR = '#d7191c'
ALPHA_SC  = 0.18
N_SAMPLE  = 50_000     # subsample for scatter/sigma plots
RNG       = np.random.default_rng(42)

def subsample(n_total, n_keep, rng):
    idx = rng.choice(n_total, size=min(n_keep, n_total), replace=False)
    idx.sort()
    return idx

# ── Pre-compute calibration curve data ───────────────────────────────────
nominal_levels = np.linspace(0.01, 0.99, 99)

def calibration_curve(errors, sigmas, levels):
    """Achieved coverage for each nominal level, using the oracle k."""
    achieved = []
    for lev in levels:
        k   = np.percentile(np.abs(errors) / np.clip(sigmas, 1e-10, None), lev * 100)
        cov = np.mean(np.abs(errors) <= k * sigmas)
        achieved.append(cov)
    return np.array(achieved)

mc_calib  = calibration_curve(mc_targets  - mc_preds,  mc_stds,  nominal_levels)
ens_calib = calibration_curve(ens_targets - ens_mean,  ens_std,  nominal_levels)

# indices for subsampling
n_total   = len(mc_preds)
idx_sc    = subsample(n_total, N_SAMPLE, RNG)

# ── FIG 1 — Predicted vs Actual ──────────────────────────────────────────
print('Generating Fig 1: Scatter predicted vs actual...')
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True)

for ax, preds, sigs, title, color in [
    (axes[0], mc_preds,  mc_stds,  'MC Dropout',   MC_COLOR),
    (axes[1], ens_mean,  ens_std,   'Deep Ensemble', ENS_COLOR),
]:
    tgt_s = mc_targets[idx_sc]
    prd_s = preds[idx_sc]
    sig_s = sigs[idx_sc]

    sc = ax.scatter(tgt_s, prd_s,
                    c=sig_s, cmap='YlOrRd', alpha=ALPHA_SC,
                    s=2, rasterized=True, linewidths=0)
    lim_lo = min(tgt_s.min(), prd_s.min()) * 1.05
    lim_hi = max(tgt_s.max(), prd_s.max()) * 1.05
    ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi],
            'k--', lw=1.2, alpha=0.7, label='Perfect prediction')

    r2_val  = mc_results['point_prediction']['r2']  if 'MC' in title else ens_results['point_prediction']['r2']
    mae_val = mc_results['point_prediction']['mae'] if 'MC' in title else ens_results['point_prediction']['mae']
    ax.text(0.05, 0.95,
            f'$R^2$ = {r2_val:.3f}\nMAE = {mae_val:.2f} veh/h',
            transform=ax.transAxes, fontsize=9,
            verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='none'))

    ax.set_xlabel('True flow change (veh/h)')
    ax.set_title(title, fontweight='bold')
    cbar = fig.colorbar(sc, ax=ax, shrink=0.75)
    cbar.set_label('$\\sigma$ (veh/h)', fontsize=8)

axes[0].set_ylabel('Predicted flow change (veh/h)')
fig.suptitle(f'Predicted vs. True Flow Change  (n = {N_SAMPLE:,} sampled nodes)', y=1.01)
fig.tight_layout()
p1 = f'{PLOT_DIR}/fig1_scatter_predicted_vs_actual.png'
fig.savefig(p1)
plt.close(fig)
print(f'  Saved: {p1}')

# ── FIG 2 — Calibration Curve ────────────────────────────────────────────
print('Generating Fig 2: Calibration curve...')
fig, ax = plt.subplots(figsize=(5.5, 4.5))

ax.plot([0, 1], [0, 1], 'k--', lw=1.2, alpha=0.6, label='Perfect calibration')
ax.plot(nominal_levels, mc_calib,  color=MC_COLOR,  lw=2, label='MC Dropout')
ax.plot(nominal_levels, ens_calib, color=ENS_COLOR, lw=2, label='Deep Ensemble')

# annotate the 1.96σ coverage points
for cov_val, color, label in [
    (mc_results['calibration']['coverage_at_1.96sigma'],  MC_COLOR,  'MC'),
    (ens_results['calibration']['coverage_at_1.96sigma'], ENS_COLOR, 'Ens'),
]:
    ax.axhline(cov_val, color=color, lw=0.8, ls=':', alpha=0.7)
    ax.text(0.01, cov_val + 0.01, f'{label}: {cov_val:.2f} @ 1.96σ',
            color=color, fontsize=8)

ax.fill_between([0, 1], [0, 1], [0, 1], alpha=0.04, color='grey', label='Overconfident region')
ax.set_xlabel('Nominal coverage level')
ax.set_ylabel('Achieved coverage')
ax.set_title('Calibration Curve — Nominal vs. Achieved Coverage', fontweight='bold')
ax.legend(loc='upper left')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_aspect('equal')
fig.tight_layout()
p2 = f'{PLOT_DIR}/fig2_calibration_curve.png'
fig.savefig(p2)
plt.close(fig)
print(f'  Saved: {p2}')

# ── FIG 3 — Sigma vs |Error| ─────────────────────────────────────────────
print('Generating Fig 3: Sigma vs |error|...')
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

for ax, preds, sigs, title, color, res in [
    (axes[0], mc_preds, mc_stds,  'MC Dropout',    MC_COLOR,  mc_results),
    (axes[1], ens_mean, ens_std,   'Deep Ensemble', ENS_COLOR, ens_results),
]:
    abs_err = np.abs(mc_targets - preds)
    sig_s   = sigs[idx_sc]
    err_s   = abs_err[idx_sc]

    ax.scatter(sig_s, err_s,
               alpha=ALPHA_SC, s=2, color=color, rasterized=True, linewidths=0)

    # ideal line: err = sigma
    xlim = np.array([0, np.percentile(sig_s, 99)])
    ax.plot(xlim, xlim, 'k--', lw=1.2, alpha=0.6, label='|error| = σ')

    rho = res['uncertainty_quality']['spearman_rho']
    ax.text(0.97, 0.05,
            f'Spearman ρ = {rho:.3f}',
            transform=ax.transAxes, fontsize=9, ha='right',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='none'))

    ax.set_xlabel('Predicted uncertainty σ (veh/h)')
    ax.set_ylabel('|Prediction error| (veh/h)')
    ax.set_title(title, fontweight='bold')
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.legend(loc='upper left', fontsize=8)

fig.suptitle(f'Uncertainty vs. Absolute Error  (n = {N_SAMPLE:,} sampled nodes)', y=1.01)
fig.tight_layout()
p3 = f'{PLOT_DIR}/fig3_sigma_vs_error.png'
fig.savefig(p3)
plt.close(fig)
print(f'  Saved: {p3}')

# ── FIG 4 — Uncertainty Distribution ─────────────────────────────────────
print('Generating Fig 4: Uncertainty distribution...')
fig, ax = plt.subplots(figsize=(5.5, 4.5))

bins = np.linspace(0, np.percentile(np.concatenate([mc_stds, ens_std]), 99.5), 80)
ax.hist(mc_stds,  bins=bins, alpha=0.5, color=MC_COLOR,
        label=f'MC Dropout  (μ={mc_results["uncertainty_quality"]["mean_sigma"]:.3f})',
        density=True, edgecolor='none')
ax.hist(ens_std,  bins=bins, alpha=0.5, color=ENS_COLOR,
        label=f'Deep Ensemble  (μ={ens_results["uncertainty_quality"]["mean_sigma"]:.3f})',
        density=True, edgecolor='none')

ax.axvline(mc_results['uncertainty_quality']['mean_sigma'],
           color=MC_COLOR,  lw=1.5, ls='--', alpha=0.9)
ax.axvline(ens_results['uncertainty_quality']['mean_sigma'],
           color=ENS_COLOR, lw=1.5, ls='--', alpha=0.9)

ax.set_xlabel('Predicted uncertainty σ (veh/h)')
ax.set_ylabel('Density')
ax.set_title('Distribution of Predicted Uncertainty', fontweight='bold')
ax.legend()
fig.tight_layout()
p4 = f'{PLOT_DIR}/fig4_uncertainty_distribution.png'
fig.savefig(p4)
plt.close(fig)
print(f'  Saved: {p4}')

print()
print('All 4 plots saved to:', PLOT_DIR)